# 09d — Diagnosa Presisi Numerik Qwen2.5-VL-3B (Kaggle, 2×T4)

**Latar.** Uji kewajaran 09s: Qwen memberi jawaban **identik** untuk citra asli, citra hitam, dan
derau, bahkan untuk pertanyaan mudah ("adakah bangunan?" dijawab YA pada citra hitam). InternVL3-2B
sebaliknya membedakan citra hitam dari citra asli. Artinya jalur citra Qwen kemungkinan
**tidak berfungsi**, bukan sekadar model tidak mampu menilai kerusakan.

**Dugaan utama:** semua run Qwen memakai **float16** karena T4 tidak mendukung bfloat16. Qwen2.5-VL
dilatih dalam bfloat16; pada float16 aktivasi encoder visual dapat **melampaui batas** (inf/NaN)
sehingga semua citra menjadi representasi yang sama.

**Uji (ditetapkan sebelum dijalankan):** untuk 3 chip (satu per kelas) + citra hitam + derau:
1. statistik embedding keluaran encoder visual: jumlah NaN/inf, norma, kemiripan kosinus antarcitra;
2. pertanyaan mudah (warna kotak; adakah bangunan) dan prompt kelas v4.

Dijalankan pada tiga konfigurasi: **fp16 (seperti run lama)**, **fp16 + attention eager**, **fp32
dibagi ke 2 GPU**. Bila fp32 membedakan citra sedangkan fp16 tidak, hasil zero-shot Qwen lama
tidak sah dan harus diulang dalam fp32.

Prasyarat: **GPU T4 x2**, Internet On, dataset chip + paket. ± 15 menit.

In [ ]:
!pip install -q "transformers==4.51.3" accelerate
import torch, transformers
print(transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "Pilih GPU T4 x2 di Notebook options"

In [ ]:
import json, gc, time, datetime, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

INPUT, KERJA = Path("/kaggle/input"), Path("/kaggle/working")
def cari_satu(pola):
    h = sorted(INPUT.rglob(pola));  assert h, pola;  return h[-1]
KONFIG = json.loads(cari_satu("konfigurasi_fitur.json").read_text(encoding="utf-8"))
MP = json.loads(cari_satu("manifes_v*_*.json").read_text(encoding="utf-8"))
PROMPT_KELAS = cari_satu(MP["prompt"]["kelas"]["berkas"]).read_text(encoding="utf-8")
spec = importlib.util.spec_from_file_location("p05", cari_satu("05_parser_keluaran.py"))
P = importlib.util.module_from_spec(spec); spec.loader.exec_module(P)
besar = pd.read_csv(cari_satu("sampel_besar.csv")); kecil = pd.read_csv(cari_satu("sampel_kecil.csv"))
calon = besar[~besar.uid.isin(kecil.uid)]
DEV = pd.concat([g.sample(5, random_state=42) for _, g in calon.groupby("kelas_xbd")])
UJI = DEV.groupby("kelas_xbd").head(1).reset_index(drop=True)
peta = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name == "pasca"}
CITRA = {f"{r.kelas_xbd}:{r.uid[:8]}": Image.open(peta[r.chip_pasca]).convert("RGB") for r in UJI.itertuples()}
CITRA["HITAM"] = Image.new("RGB", (448, 448), (0, 0, 0))
CITRA["DERAU"] = Image.fromarray(np.random.default_rng(42).integers(0, 256, (448, 448, 3), dtype=np.uint8))
TANYA = {"warna": "Perhatikan kotak berwarna di tengah citra ini. Apa warna kotak itu? Jawab satu kata.",
         "ada": "Apakah di dalam kotak di tengah citra ini terdapat bangunan? Jawab hanya YA atau TIDAK.",
         "kelas": PROMPT_KELAS}
REPO = "Qwen/Qwen2.5-VL-3B-Instruct"
proc = AutoProcessor.from_pretrained(REPO)
print(list(CITRA))

In [ ]:
def uji_konfigurasi(nama, **kw):
    t0 = time.time()
    m = Qwen2_5_VLForConditionalGeneration.from_pretrained(REPO, **kw).eval()
    dev_vis = next(m.visual.parameters()).device; dt = next(m.visual.parameters()).dtype
    dev_in = m.get_input_embeddings().weight.device
    emb, jawab = {}, []
    for k, img in CITRA.items():
        pesan = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": "x"}]}]
        x = proc(text=[proc.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)],
                 images=[img], return_tensors="pt")
        with torch.inference_mode():
            e = m.visual(x["pixel_values"].to(dev_vis, dt), grid_thw=x["image_grid_thw"].to(dev_vis)).float()
        emb[k] = e
        for q, teks in TANYA.items():
            pesan = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": teks}]}]
            x = proc(text=[proc.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)],
                     images=[img], return_tensors="pt").to(dev_in)
            with torch.inference_mode():
                y = m.generate(**x, max_new_tokens=40 if q == "kelas" else 8, do_sample=False)
            t = proc.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            if q == "kelas":
                t = P.urai_kelas(t, KONFIG)["kelas_kerusakan"]
            jawab.append({"konfigurasi": nama, "citra": k, "pertanyaan": q, "jawab": t[:30]})
    stat = pd.DataFrame([{"konfigurasi": nama, "citra": k,
                          "n_nan": int(torch.isnan(e).sum()), "n_inf": int(torch.isinf(e).sum()),
                          "norma_rerata": round(float(torch.nan_to_num(e).norm(dim=-1).mean()), 2)}
                         for k, e in emb.items()])
    rata = {k: torch.nan_to_num(e).mean(0) for k, e in emb.items()}
    ks = list(rata)
    kos = pd.DataFrame([[round(float(torch.nn.functional.cosine_similarity(rata[a], rata[b], dim=0)), 3)
                         for b in ks] for a in ks], index=ks, columns=ks)
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"\n######## {nama}  ({time.time()-t0:.0f} s)")
    print(stat.to_string(index=False))
    print("kemiripan kosinus rerata embedding visual antarcitra:\n" + kos.to_string())
    j = pd.DataFrame(jawab).pivot(index="citra", columns="pertanyaan", values="jawab")
    print(j.to_string())
    return stat, kos, j

HASIL = {}
HASIL["fp16"] = uji_konfigurasi("fp16", torch_dtype=torch.float16, device_map={"": 0})
HASIL["fp16_eager"] = uji_konfigurasi("fp16_eager", torch_dtype=torch.float16, device_map={"": 0},
                                      attn_implementation="eager")
HASIL["fp32_2gpu"] = uji_konfigurasi("fp32_2gpu", torch_dtype=torch.float32, device_map="auto",
                                     max_memory={0: "13GiB", 1: "14GiB"})

In [ ]:
# Putusan otomatis: konfigurasi 'melihat citra' bila jawaban citra HITAM berbeda dari citra asli
# pada pertanyaan 'ada' atau 'warna', dan tidak ada NaN/inf pada embedding visual.
for k, (stat, kos, j) in HASIL.items():
    asli = [c for c in j.index if ":" in c]
    beda = any(j.loc["HITAM", q] != j.loc[c, q] for c in asli for q in ("ada", "warna"))
    rusak = int(stat.n_nan.sum() + stat.n_inf.sum())
    kos_asli_hitam = float(np.mean([kos.loc[c, "HITAM"] for c in asli]))
    print(f"{k:<11} NaN/inf={rusak:<6} kosinus(asli,HITAM)={kos_asli_hitam:.3f}  "
          f"jawaban HITAM beda dari asli={beda}  ->  {'MELIHAT' if beda and not rusak else 'BUTA/RUSAK'}")

In [ ]:
import shutil
RUN_ID = f"{datetime.datetime.now(datetime.timezone.utc):%Y%m%d-%H%M%S}_diagnosa-presisi-qwen"
DIR = KERJA / "runs" / RUN_ID; DIR.mkdir(parents=True)
teks = []
for k, (stat, kos, j) in HASIL.items():
    teks += [f"######## {k}", stat.to_string(index=False), "kosinus:", kos.to_string(), j.to_string(), ""]
(DIR / "ringkasan.txt").write_text("\n".join(teks), encoding="utf-8")
(DIR / "manifest.json").write_text(json.dumps({"run_id": RUN_ID, "jenis": "kewajaran",
    "tujuan": "diagnosa presisi numerik encoder visual Qwen2.5-VL-3B (fp16 vs fp32)",
    "citra": list(CITRA), "transformers": transformers.__version__, "torch": torch.__version__,
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}, indent=2), encoding="utf-8")
print("arsip:", shutil.make_archive(str(KERJA / f"run_{RUN_ID}"), "zip", root_dir=KERJA / "runs", base_dir=RUN_ID))